In [1]:
from matplotlib.backends.backend_pdf import PdfPages
import argparse
import pathlib


import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import json

import sys
from pathlib import Path

In [2]:
def evaluate_model(
    model_path,
    test_path,      # validation set (for optimization)
    test_path2,     # test set (for plotting)
    p_edges,
    theta_edges,
    n_thresholds=100,
):
    import numpy as np
    import pandas as pd
    import pathlib
    import joblib
    import json

    # ─────────────────────────────────────────────
    # output directory
    # ─────────────────────────────────────────────
    outdir = pathlib.Path(
        "/work/clas12/CooperBe/Argonne2026/suli2026_pid/figures/optimized/"
    )
    outdir.mkdir(parents=True, exist_ok=True)

    out_csv = outdir / "optimized_bdt_thresholds.csv"

    # ─────────────────────────────────────────────
    # load model
    # ─────────────────────────────────────────────
    _model_raw = joblib.load(str(model_path))

    if isinstance(_model_raw, dict) and "model" in _model_raw:
        model = _model_raw["model"]
        feature_names = _model_raw["features"]
    else:
        model = _model_raw
        manifest_path = pathlib.Path(test_path).resolve().parents[0] / "manifest.json"

        with open(manifest_path, "r") as f:
            manifest = json.load(f)

        feature_names = manifest.get("feature_list") or manifest.get("columns")

    # ─────────────────────────────────────────────
    # load VALIDATION set (for threshold optimization)
    # ─────────────────────────────────────────────
    df = pd.read_parquet(str(test_path))

    X = df[feature_names].to_numpy(dtype=np.float32)
    df = df.copy()
    df["score"] = model.predict_proba(X)[:, 1]

    # ─────────────────────────────────────────────
    # threshold grid
    # ─────────────────────────────────────────────
    thresholds = np.linspace(0.0, 0.95, n_thresholds)

    results = []

    n_p = len(p_edges) - 1
    n_t = len(theta_edges) - 1

    # ─────────────────────────────────────────────
    # BIN LOOP (optimize threshold per bin)
    # ─────────────────────────────────────────────
    for ti in range(n_t):
        theta_lo = theta_edges[ti]
        theta_hi = theta_edges[ti + 1]

        for pi in range(n_p):
            p_lo = p_edges[pi]
            p_hi = p_edges[pi + 1]

            df_bin = df[
                (df["p"] >= p_lo) & (df["p"] < p_hi) &
                (df["theta"] >= theta_lo) & (df["theta"] < theta_hi)
            ]

            if len(df_bin) == 0:
                results.append({
                    "p_lo": p_lo,
                    "p_hi": p_hi,
                    "theta_lo": theta_lo,
                    "theta_hi": theta_hi,
                    "best_threshold": np.nan,
                    "best_fom": np.nan,
                })
                continue

            mc = df_bin["mc_matching_pid"].to_numpy()
            scores = df_bin["score"].to_numpy()

            is_K = (mc == 321)
            is_pi = (mc == 211)

            if np.sum(is_K) == 0:
                results.append({
                    "p_lo": p_lo,
                    "p_hi": p_hi,
                    "theta_lo": theta_lo,
                    "theta_hi": theta_hi,
                    "best_threshold": np.nan,
                    "best_fom": np.nan,
                })
                continue

            best_t = np.nan
            best_fom = -np.inf

            for t in thresholds:
                accepted = scores > t

                N_K = np.sum(accepted & is_K)
                N_pi = np.sum(accepted & is_pi)

                denom = np.sqrt(N_K + N_pi)

                if denom == 0:
                    continue

                fom = N_K / denom

                if fom > best_fom:
                    best_fom = fom
                    best_t = t

            results.append({
                "p_lo": p_lo,
                "p_hi": p_hi,
                "theta_lo": theta_lo,
                "theta_hi": theta_hi,
                "best_threshold": best_t,
                "best_fom": best_fom,
            })

    threshold_df = pd.DataFrame(results)

    # ─────────────────────────────────────────────
    # SAVE (THIS WAS MISSING BEFORE)
    # ─────────────────────────────────────────────
    threshold_df.to_csv(out_csv, index=False)
    print(f"Saved thresholds → {out_csv}")

    # ─────────────────────────────────────────────
    # load TEST set (for plotting only)
    # ─────────────────────────────────────────────
    test_df = pd.read_parquet(str(test_path2))
    X_test = test_df[feature_names].to_numpy(dtype=np.float32)
    test_df = test_df.copy()
    test_df["score"] = model.predict_proba(X_test)[:, 1]
    # ─────────────────────────────────────────────
    # PLOT using optimized thresholds
    # ─────────────────────────────────────────────
    plot_ml_contamination_simple(
        matched=test_df,
        threshold_df=threshold_df,
        pStart=0.5,
        pEnd=3.2,
        pStep=(3.2 - 0.5) / 10,
        direct=str(outdir) + "/"
    )
    plot_ml_efficiency_simple(
        matched=test_df,
        threshold_df=threshold_df,
        pStart=0.5,
        pEnd=3.2,
        pStep=(3.2 - 0.5) / 10,
        direct=str(outdir) + "/"
    )
    p_edges_diag = np.linspace(0.5, 3.2, 4)   # 3 bins → 4 edges
    theta_edges_diag = np.linspace(5, 35, 4)  # 3 bins → 4 edges

    plot_fom_vs_t(
        df=test_df,
        p_edges=p_edges_diag,
        theta_edges=theta_edges_diag,
        n_thresholds=n_thresholds
    )
    print(f"Done")
        

    return threshold_df, test_df

In [3]:
def plot_ml_contamination_simple(
    matched,
    threshold_df,
    pStart,
    pEnd,
    pStep,
    direct
):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    mkf=matched[matched["mc_matching_pid"]!=-9999]
    matched=mkf
    fig, ax = plt.subplots()

    t_edges = np.linspace(5, 35, 6)
    colors = ["black", "tab:blue", "tab:orange", "tab:green", "tab:red"]

    n_p_bins = int((pEnd - pStart) / pStep)

    # ---------------------------------------------------------
    # loop over theta bins
    # ---------------------------------------------------------
    for i in range(len(t_edges) - 1):

        theta_lo = t_edges[i]
        theta_hi = t_edges[i + 1]

        vals = []
        errs = []

        # -----------------------------------------------------
        # loop over p bins
        # -----------------------------------------------------
        for j in range(n_p_bins):

            p_lo = pStart + j * pStep
            p_hi = p_lo + pStep

            # -------------------------------------------------
            # get bin-specific threshold
            # -------------------------------------------------
            row = threshold_df[
                (threshold_df["p_lo"] == p_lo) &
                (threshold_df["p_hi"] == p_hi) &
                (threshold_df["theta_lo"] == theta_lo) &
                (threshold_df["theta_hi"] == theta_hi)
            ]

            if len(row) == 0 or np.isnan(row["best_threshold"].values[0]):
                vals.append(np.nan)
                errs.append(np.nan)
                continue

            bdt_cut = row["best_threshold"].values[0]

            # -------------------------------------------------
            # apply cut
            # -------------------------------------------------
            pCut = matched[
                (matched["p"] >= p_lo) &
                (matched["p"] < p_hi) &
                (matched["theta"] >= theta_lo) &
                (matched["theta"] < theta_hi) &
                (matched["score"] > bdt_cut)
            ]

            a = ((pCut["mc_matching_pid"] != 321) &
                 (pCut["pid"] == 321)).sum()

            b = (pCut["pid"] == 321).sum()

            if b != 0:
                r = a / b
                rErr = r * np.sqrt((1/a if a > 0 else 0) + (1/b))
            else:
                r = np.nan
                rErr = np.nan

            vals.append(r)
            errs.append(rErr)

        # -----------------------------------------------------
        # plotting
        # -----------------------------------------------------
        edges = np.linspace(pStart, pEnd, len(vals) + 1)
        x = (edges[:-1] + edges[1:]) / 2

        vals = np.array(vals)
        errs = np.array(errs)

        mask = ~np.isnan(vals)

        ax.errorbar(
            x[mask],
            vals[mask],
            yerr=errs[mask],
            fmt='o',
            capsize=3,
            color=colors[i],
            label=f"{theta_lo:.0f}–{theta_hi:.0f}°"
        )

    ax.set_ylim(0, 1.1)
    ax.set_xlabel("Momentum (GeV/c)")
    ax.set_ylabel("Contamination")
    ax.set_title("ML-based K⁺ contamination (bin-optimized BDT cuts)")

    ax.legend()

    fig.tight_layout()
    fig.savefig(direct + "contaminationK_ML.png", dpi=150)
    plt.close(fig)

    return fig

In [4]:
def plot_ml_efficiency_simple(
    matched,
    threshold_df,
    pStart,
    pEnd,
    pStep,
    direct
):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    mkf = matched[matched["mc_matching_pid"] != -9999]
    matched = mkf

    fig, ax = plt.subplots()

    t_edges = np.linspace(5, 35, 6)
    colors = ["black", "tab:blue", "tab:orange", "tab:green", "tab:red"]

    n_p_bins = int((pEnd - pStart) / pStep)

    # ---------------------------------------------------------
    # loop over theta bins
    # ---------------------------------------------------------
    for i in range(len(t_edges) - 1):

        theta_lo = t_edges[i]
        theta_hi = t_edges[i + 1]

        vals = []
        errs = []

        # -----------------------------------------------------
        # loop over p bins
        # -----------------------------------------------------
        for j in range(n_p_bins):

            p_lo = pStart + j * pStep
            p_hi = p_lo + pStep

            # -------------------------------------------------
            # get bin-specific threshold
            # -------------------------------------------------
            row = threshold_df[
                (threshold_df["p_lo"] == p_lo) &
                (threshold_df["p_hi"] == p_hi) &
                (threshold_df["theta_lo"] == theta_lo) &
                (threshold_df["theta_hi"] == theta_hi)
            ]

            if len(row) == 0 or np.isnan(row["best_threshold"].values[0]):
                vals.append(np.nan)
                errs.append(np.nan)
                continue

            bdt_cut = row["best_threshold"].values[0]

            # -------------------------------------------------
            # apply cut
            # -------------------------------------------------
            pCut = matched[
                (matched["p"] >= p_lo) &
                (matched["p"] < p_hi) &
                (matched["theta"] >= theta_lo) &
                (matched["theta"] < theta_hi) &
                (matched["score"] > bdt_cut)
            ]

            # =================================================
            # EFFICIENCY (CHANGED PART ONLY)
            # =================================================
            true_K = (pCut["mc_matching_pid"] == 321).sum()
            total_K = ((matched["p"] >= p_lo) &
                       (matched["p"] < p_hi) &
                       (matched["theta"] >= theta_lo) &
                       (matched["theta"] < theta_hi) &
                       (matched["mc_matching_pid"] == 321)).sum()

            if total_K != 0:
                r = true_K / total_K
                rErr = np.sqrt(r * (1 - r) / total_K)
            else:
                r = np.nan
                rErr = np.nan

            vals.append(r)
            errs.append(rErr)

        # -----------------------------------------------------
        # plotting
        # -----------------------------------------------------
        edges = np.linspace(pStart, pEnd, len(vals) + 1)
        x = (edges[:-1] + edges[1:]) / 2

        vals = np.array(vals)
        errs = np.array(errs)

        mask = ~np.isnan(vals)

        ax.errorbar(
            x[mask],
            vals[mask],
            yerr=errs[mask],
            fmt='o',
            capsize=3,
            color=colors[i],
            label=f"{theta_lo:.0f}–{theta_hi:.0f}°"
        )

    ax.set_ylim(0, 1.1)
    ax.set_xlabel("Momentum (GeV/c)")
    ax.set_ylabel("Efficiency")
    ax.set_title("BDT efficiency (bin-optimized cuts)")

    ax.legend()

    fig.tight_layout()
    fig.savefig(direct + "efficiencyK_ML.png", dpi=150)
    plt.close(fig)

    return fig

In [5]:
import sklearn
import sys


print("sklearn:", sklearn.__version__)
print("python:", sys.executable)

sklearn: 1.5.2
python: /opt/conda/bin/python


In [6]:
import numpy as np
import matplotlib.pyplot as plt

def plot_fom_vs_t(df, p_edges, theta_edges, n_thresholds=100):

    thresholds = np.linspace(0.0, 0.95, n_thresholds)

    n_p = len(p_edges) - 1
    n_t = len(theta_edges) - 1

    for ti in range(n_t):
        theta_lo = theta_edges[ti]
        theta_hi = theta_edges[ti + 1]

        for pi in range(n_p):
            p_lo = p_edges[pi]
            p_hi = p_edges[pi + 1]

            df_bin = df[
                (df["p"] >= p_lo) & (df["p"] < p_hi) &
                (df["theta"] >= theta_lo) & (df["theta"] < theta_hi)
            ]

            if len(df_bin) == 0:
                continue

            mc = df_bin["mc_matching_pid"].to_numpy()
            scores = df_bin["score"].to_numpy()

            is_K = (mc == 321)
            is_pi = (mc == 211)

            fom_vals = []

            for t in thresholds:
                accepted = scores > t

                N_K = np.sum(accepted & is_K)
                N_pi = np.sum(accepted & is_pi)

                denom = np.sqrt(N_K + N_pi)

                fom_vals.append(N_K / denom if denom > 0 else np.nan)

            # optional: find optimum just for sanity check
            best_idx = np.nanargmax(fom_vals)
            best_t = thresholds[best_idx]

            plt.figure()
            plt.plot(thresholds, fom_vals)
            plt.axvline(best_t, linestyle="--")
            plt.title(
                f"FOM(t)\n"
                f"p[{p_lo:.2f},{p_hi:.2f}]  θ[{theta_lo:.1f},{theta_hi:.1f}]"
            )
            plt.xlabel("BDT threshold t")
            plt.ylabel("FOM(t)")
            plt.grid()
            plt.show()

In [7]:
threshold_df, test_df = evaluate_model(
    model_path="/volatile/clas12/cooperb/SULI/tier2All/model_v01/model.joblib",
    test_path="/volatile/clas12/cooperb/SULI/dataset_v02/val.parquet",
    test_path2="/volatile/clas12/cooperb/SULI/dataset_v02/test.parquet",
    p_edges=np.linspace(0.5, 3.2, 11),
    theta_edges=np.linspace(5, 35, 6),
    n_thresholds=100
)

Saved thresholds → /work/clas12/CooperBe/Argonne2026/suli2026_pid/figures/optimized/optimized_bdt_thresholds.csv
Done
